# C2 Dispatch Analysis

Loads `c2_ccp_r1.xlsx` and `c2_r1_r2.xlsx` from a single-run folder and
analyses dispatch decisions:

1. **Ambulance dispatch frequency** — how many dispatches per C2 level
2. **Wake-up triggers** — what caused each dispatch episode
3. **T1 unserved** — T1 requests that found no ambulance
4. **T2 ignored** — T2 requests present but not assigned
5. **T1/T2 cargo histogram** — load composition for urgent transports
6. **T3 cargo distribution** — how many T3s per trip

In [ ]:
import sys
from pathlib import Path
_root = Path(".").resolve()
while not (_root / "paths.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
import paths

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

run_folder = paths.SINGLE_RUN_OUTPUT / FOLDER_NAME

def load_c2(name):
    path = run_folder / f"{name}.xlsx"
    if not path.exists():
        print(f"{name}.xlsx not found")
        return pd.DataFrame()
    df = pd.read_excel(path)
    # Parse JSON columns if they exist as strings
    for col in ["transport_requests", "available_ambulances", "assignments"]:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    if "event" in df.columns and df["event"].dtype == object:
        df["event"] = df["event"].apply(lambda x: json.loads(x) if isinstance(x, str) and x.startswith('"') else x)
    return df

c2_ccp = load_c2("c2_ccp_r1")
c2_r1  = load_c2("c2_r1_r2")
print(f"CCP→R1: {len(c2_ccp)} dispatch rows  |  R1→R2: {len(c2_r1)} dispatch rows")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HELPER: extract per-row stats from old-format logs (JSON-only columns)
# New-format logs have flat columns (time, trigger, n_t1_cargo, etc.).
# This cell normalises both formats into a common shape.
# ═══════════════════════════════════════════════════════════════════════════

def enrich(df):
    """Add derived columns if not already present (old-format logs)."""
    if df.empty:
        return df
    out = df.copy()

    if "n_requests" not in out.columns:
        out["n_requests"] = out["transport_requests"].apply(lambda x: len(x) if isinstance(x, list) else 0)
    if "n_idle_ambulances" not in out.columns:
        out["n_idle_ambulances"] = out["available_ambulances"].apply(lambda x: len(x) if isinstance(x, list) else 0)

    if "n_assigned" not in out.columns:
        def _count_assigned(row):
            a = row["assignments"]
            if isinstance(a, dict):
                return sum(len(v) for v in a.values())
            return 0
        out["n_assigned"] = out.apply(_count_assigned, axis=1)

    if "n_t1_cargo" not in out.columns:
        def _cargo(row):
            a = row["assignments"]
            reqs = row["transport_requests"]
            if not isinstance(a, dict) or not isinstance(reqs, list):
                return (0, 0, 0)
            assigned_ids = {pid for pids in a.values() for pid in pids}
            triage_of = {r["patient_id"]: r["triage"] for r in reqs}
            triages = [triage_of.get(pid, "") for pid in assigned_ids]
            return (
                sum(1 for t in triages if t.startswith("T1")),
                sum(1 for t in triages if t.startswith("T2")),
                sum(1 for t in triages if t.startswith("T3")),
            )
        cargo = out.apply(_cargo, axis=1, result_type="expand")
        cargo.columns = ["n_t1_cargo", "n_t2_cargo", "n_t3_cargo"]
        out = pd.concat([out, cargo], axis=1)

    if "trigger" not in out.columns:
        out["trigger"] = "unknown"

    return out

c2_ccp = enrich(c2_ccp)
c2_r1  = enrich(c2_r1)
print("Enriched.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1. DISPATCH FREQUENCY
# ═══════════════════════════════════════════════════════════════════════════

for label, df in [("CCP → R1", c2_ccp), ("R1 → R2", c2_r1)]:
    n_dispatched = (df["n_assigned"] > 0).sum()
    n_empty      = (df["n_assigned"] == 0).sum()
    total_patients = df["n_assigned"].sum()
    print(f"{label}:  {n_dispatched} dispatches ({total_patients} patients)  |  "
          f"{n_empty} failed (no ambulance or no actionable request)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2. WAKE-UP TRIGGERS
# ═══════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, df) in zip(axes, [("CCP → R1", c2_ccp), ("R1 → R2", c2_r1)]):
    if "trigger" in df.columns:
        counts = df["trigger"].value_counts()
    else:
        counts = df["event"].value_counts()
    counts.plot.bar(ax=ax, edgecolor="black")
    ax.set_title(f"{label} — dispatch triggers")
    ax.set_ylabel("Count")
    for i, v in enumerate(counts):
        ax.text(i, v + 0.5, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3. T1 UNSERVED — T1 in requests but nobody assigned
# ═══════════════════════════════════════════════════════════════════════════

for label, df in [("CCP → R1", c2_ccp), ("R1 → R2", c2_r1)]:
    t1_unserved = 0
    t1_total = 0
    for _, row in df.iterrows():
        reqs = row["transport_requests"]
        assigns = row["assignments"]
        if not isinstance(reqs, list):
            continue
        assigned_ids = set()
        if isinstance(assigns, dict):
            for pids in assigns.values():
                assigned_ids.update(pids)
        for r in reqs:
            if str(r.get("triage", "")).startswith("T1"):
                t1_total += 1
                if r["patient_id"] not in assigned_ids:
                    t1_unserved += 1
    print(f"{label}:  T1 requests: {t1_total}  |  "
          f"T1 not assigned in that row: {t1_unserved}  "
          f"({100*t1_unserved/t1_total:.1f}%)" if t1_total else f"{label}: no T1 requests")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 4. T2 IGNORED — T2 in requests but not assigned
# ═══════════════════════════════════════════════════════════════════════════

for label, df in [("CCP → R1", c2_ccp), ("R1 → R2", c2_r1)]:
    t2_ignored = 0
    t2_total = 0
    for _, row in df.iterrows():
        reqs = row["transport_requests"]
        assigns = row["assignments"]
        if not isinstance(reqs, list):
            continue
        assigned_ids = set()
        if isinstance(assigns, dict):
            for pids in assigns.values():
                assigned_ids.update(pids)
        for r in reqs:
            if str(r.get("triage", "")).startswith("T2"):
                t2_total += 1
                if r["patient_id"] not in assigned_ids:
                    t2_ignored += 1
    print(f"{label}:  T2 requests: {t2_total}  |  "
          f"T2 not assigned in that row: {t2_ignored}  "
          f"({100*t2_ignored/t2_total:.1f}%)" if t2_total else f"{label}: no T2 requests")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 5. T1/T2 CARGO HISTOGRAM
# ═══════════════════════════════════════════════════════════════════════════
# Only rows where at least one T1 or T2 was loaded.

for label, df in [("CCP → R1", c2_ccp), ("R1 → R2", c2_r1)]:
    mask = (df["n_t1_cargo"] + df["n_t2_cargo"]) > 0
    sub = df.loc[mask].copy()
    if sub.empty:
        print(f"{label}: no T1/T2 dispatches")
        continue

    sub["cargo_label"] = sub.apply(
        lambda r: f"{int(r['n_t1_cargo'])}×T1 + {int(r['n_t2_cargo'])}×T2"
                  if r["n_t1_cargo"] > 0 and r["n_t2_cargo"] > 0
                  else (f"{int(r['n_t1_cargo'])}×T1" if r["n_t1_cargo"] > 0
                        else f"{int(r['n_t2_cargo'])}×T2"),
        axis=1,
    )
    counts = sub["cargo_label"].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(8, 4))
    counts.plot.bar(ax=ax, edgecolor="black", color="#3498db")
    ax.set_title(f"{label} — T1/T2 cargo composition")
    ax.set_ylabel("Dispatches")
    ax.set_xlabel("Cargo")
    for i, v in enumerate(counts):
        ax.text(i, v + 0.3, str(v), ha="center", fontweight="bold")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 6. T3 CARGO DISTRIBUTION
# ═══════════════════════════════════════════════════════════════════════════
# Only rows where at least one T3 was loaded.

for label, df in [("CCP → R1", c2_ccp), ("R1 → R2", c2_r1)]:
    sub = df.loc[df["n_t3_cargo"] > 0].copy()
    if sub.empty:
        print(f"{label}: no T3 dispatches")
        continue

    fig, ax = plt.subplots(figsize=(6, 4))
    t3_counts = sub["n_t3_cargo"].astype(int).value_counts().sort_index()
    t3_counts.plot.bar(ax=ax, edgecolor="black", color="#2ecc71")
    ax.set_title(f"{label} — T3 patients per dispatch")
    ax.set_xlabel("Number of T3s transported")
    ax.set_ylabel("Dispatches")
    for i, v in enumerate(t3_counts):
        ax.text(i, v + 0.3, str(v), ha="center", fontweight="bold")
    plt.tight_layout()
    plt.show()